# Elpris SE3 – Datainhämtning

In [1]:
import sys
import time
from datetime import date, timedelta
from pathlib import Path

import requests
import pandas as pd

## 1. Utforska API:t

In [2]:
url = "https://www.elprisetjustnu.se/api/v1/prices/2026/10-05_SE3.json"
response = requests.get(url, timeout=10)
print(response.status_code)

df = pd.DataFrame(response.json())
print(len(df))

df.head()


200
96


,SEK_per_kWh,EUR_per_kWh,EXR,time_start,time_end
0,0.05664,0.00501,11.304508,2026-10-05T00:00:00+02:00,2026-10-05T00:15:00+02:00
1,0.05596,0.00495,11.304508,2026-10-05T00:15:00+02:00,2026-10-05T00:30:00+02:00
2,0.03391,0.00300,11.304508,2026-10-05T00:30:00+02:00,2026-10-05T00:45:00+02:00
3,0.03064,0.00271,11.304508,2026-10-05T00:45:00+02:00,2026-10-05T01:00:00+02:00
4,0.01063,0.00094,11.304508,2026-10-05T01:00:00+02:00,2026-10-05T01:15:00+02:00


In [3]:
# Kontroll: SEK_per_kWh = EUR_per_kWh * EXR → EUR och EXR behövs inte
df["SEK_check"] = df["EUR_per_kWh"] * df["EXR"]
(df["SEK_check"] - df["SEK_per_kWh"]).abs().max()

np.float64(4.950079999996304e-06)

## 2. build_url

In [4]:
def build_url(day: date, area: str = "SE3") -> str:
    return (
        "https://www.elprisetjustnu.se/api/v1/prices/"
        f"{day.year}/{day.month:02d}-{day.day:02d}_{area}.json"
    )

print(build_url(date(2025, 9, 30)))

https://www.elprisetjustnu.se/api/v1/prices/2025/09-30_SE3.json


In [5]:
# Timpriser (24), sommartid→vintertid (100), äldre data
test_days = [date(2025, 9, 30), date(2025, 10, 26), date(2022, 10, 31), date(2022, 6, 15)]

for day in test_days:
    response = requests.get(build_url(day), timeout=10)
    print(f"Date: {day} | Status code: {response.status_code}")
    if response.status_code == 200:
        print(f"Row count: {len(response.json())}")

Date: 2025-09-30 | Status code: 200
Row count: 24
Date: 2025-10-26 | Status code: 200
Row count: 100
Date: 2022-10-31 | Status code: 200
Row count: 24
Date: 2022-06-15 | Status code: 200
Row count: 24


## 3. fetch_day

In [6]:
def fetch_day(day: date, area: str = "SE3") -> list[dict] | None:
    url = build_url(day, area)
    response = requests.get(url, timeout=10)

    if response.status_code == 404:
        return None

    response.raise_for_status()
    return response.json()

print(len(fetch_day(date(2025, 9, 30))))

24


In [7]:
# Hitta första dagen med data
check_days = [date(2021, 10, d) for d in range(1, 32)] + [date(2021, 11, 1)]

for day in check_days:
    result = fetch_day(day)
    if result is not None:
        print(f"Första dag med data: {day} ({len(result)} rader)")
        break
else:
    print("Ingen dag i listan hade data")

Första dag med data: 2021-11-01 (24 rader)


## 4. fetch_range

In [9]:
def fetch_range(start: date, end: date, area: str = "SE3") -> pd.DataFrame:
    rows = []
    day = start
    while day <= end:
        if day.day == 1:
            print(f"Hämtar {day:%Y-%m} ...")
        result = fetch_day(day, area)
        if result is not None:
            rows.extend(result)
        day += timedelta(days=1)
        time.sleep(0.2)
    return pd.DataFrame(rows)

df_test = fetch_range(date(2025, 9, 28), date(2025, 10, 4))
print(len(df_test))  # förväntat: 3*24 + 4*96 = 456

Hämtar 2025-10 ...
456


## 5. Ladda ner all data och spara i data/raw

In [10]:
DATA_DIR = Path("../data")
RAW_DIR = DATA_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)
raw_file = RAW_DIR / "se3_raw.csv"

start = date(2021, 11, 1)
end = date.today() - timedelta(days=1)

if raw_file.exists():
    print("Filen finns redan, laddar från disk")
    df_raw = pd.read_csv(raw_file)
else:
    df_raw = fetch_range(start, end)
    df_raw.to_csv(raw_file, index=False)
    print(f"Sparade {len(df_raw)} rader till {raw_file}")

Filen finns redan, laddar från disk


In [11]:
hourly_days = (date(2025, 9, 30) - start).days + 1
quarter_days = (end - date(2025, 10, 1)).days + 1
expected = hourly_days * 24 + quarter_days * 96
print(expected, len(df_raw), len(df_raw) - expected)

70512 70510 -2


## 6. Kontrollera saknade timmar

In [12]:
df_raw["time_start"] = pd.to_datetime(df_raw["time_start"], utc=True).dt.tz_convert("Europe/Stockholm")
df_raw["date"] = df_raw["time_start"].dt.date

counts = df_raw.groupby("date").size()
counts.value_counts()

24     1422
96      375
23        5
25        3
100       1
92        1
Name: count, dtype: int64

In [13]:
counts[counts == 23]

date
2021-11-04    23
2022-03-27    23
2023-03-26    23
2024-03-31    23
2025-03-30    23
dtype: int64

In [14]:
suspect = date.fromisoformat("2021-11-04")
df_raw.loc[df_raw["date"] == suspect, "time_start"].dt.hour.tolist()

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22]

In [15]:
len(fetch_day(date(2021, 11, 4)))

23

## 7. Kvartspriser → timpriser

In [16]:
df_raw["hour"] = (
    df_raw["time_start"]
    .dt.tz_convert("UTC")
    .dt.floor("h")
    .dt.tz_convert("Europe/Stockholm")
)

hourly = (
    df_raw.groupby("hour")["SEK_per_kWh"]
    .mean()
    .reset_index()
    .rename(columns={"hour": "time_start"})
)

print(len(hourly))
hourly.tail()

43366


,time_start,SEK_per_kWh
43361,2026-10-12 19:00:00+02:00,2.045878
43362,2026-10-12 20:00:00+02:00,1.715850
43363,2026-10-12 21:00:00+02:00,1.523707
43364,2026-10-12 22:00:00+02:00,1.460943
43365,2026-10-12 23:00:00+02:00,1.408395


In [17]:
hourly["date"] = hourly["time_start"].dt.date
hourly.groupby("date").size().value_counts()

24    1797
23       6
25       4
Name: count, dtype: int64

In [18]:
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

hourly[["time_start", "SEK_per_kWh"]].to_csv(PROCESSED_DIR / "se3_hourly.csv", index=False)
print("Sparad:", PROCESSED_DIR / "se3_hourly.csv")

Sparad: ..\data\processed\se3_hourly.csv


## 8. Dagsvärden: medel, max, min

In [19]:
daily = (
    hourly.groupby("date")["SEK_per_kWh"]
    .agg(["mean", "max", "min"])
    .reset_index()
)

print(len(daily))
daily.tail()

1807


,date,mean,max,min
1802,2026-10-08,1.291076,1.976380,0.305913
1803,2026-10-09,0.338855,0.763910,0.151062
1804,2026-10-10,0.358729,0.801017,0.143662
1805,2026-10-11,0.885049,1.590175,0.453885
1806,2026-10-12,1.404023,2.215413,0.313758


In [20]:
daily.to_csv(PROCESSED_DIR / "se3_daily.csv", index=False)
print("Sparad:", PROCESSED_DIR / "se3_daily.csv")

Sparad: ..\data\processed\se3_daily.csv


In [21]:
print(len(daily))  # förväntat: 1806
print(((daily["min"] <= daily["mean"]) & (daily["mean"] <= daily["max"])).all())  # förväntat: True
daily.loc[daily["max"].idxmax()]

1807
True


date    2022-08-30
mean      4.883027
max        8.56048
min        0.21027
Name: 302, dtype: object